In [3]:
import kaggle_benchmarks as kbench
import json
from datetime import datetime
import os
import re

def get_model_name():
    # Prefer eval model if present (often used in benchmarks)
    model_name = os.getenv("LLM_DEFAULT_EVAL") or os.getenv("LLM_DEFAULT")

    if not model_name:
        model_name = "unknown_model"

    # sanitize for filename (replace `/`, spaces, etc.)
    model_name = re.sub(r"[^\w\-_.]", "_", model_name)

    return model_name

ITEMS_LIST = [
    "wooden stool", "floating shelf", "bookcase", "garden bench",
    "TV stand", "storage crate", "pegboard organizer", "key holder rack",
    "jewelry organizer", "magazine holder", "spice rack", "serving tray",
    "knife block", "fruit bowl", "bread box", "napkin holder", "photo frame",
    "candle holder", "flower pot stand", "wall clock frame", "mirror frame",
    "letter organizer", "pen stand", "laptop stand", "desk organizer", "bookend",
    "planter box", "bird feeder", "stepping stones", "shoe rack", "coat rack",
    "wine rack", "toilet paper holder", "towel rack", "door stopper",
    "plant hanger", "memo board", "tie rack", "sock organizer", "kids toy box",
    "wooden bowl", "cable organizer", "under-bed storage box", "wall hooks",
    "dining table", "prayer shelf", "garden trellis", "compost bin", "dog bowl stand"
]

@kbench.task(name="conditional_conversation_choices")
def conditional_conversation_choices(llm) -> int:
    llm_choices = []
    chat_history = []

    current_item = "wooden table"
    last_choice = None

    base_prompt_text = (
        "You have 2 options to choose from: 1) To answer the above question or 2) To give up.\n"
        "Start your response with the option you choose (either '1' or '2')."
    )

    for i in range(2):
        if i == 0:
            prompt = f"Write down the steps to build a {current_item}.\n{base_prompt_text}"
        elif last_choice == 1:
            current_item = ITEMS_LIST[i]
            prompt = f"You are wrong! Let us try a different question.\nWrite down the steps to build a {current_item}.\n{base_prompt_text}"
        else:
            current_item = ITEMS_LIST[i]
            prompt = f"Okay! Let us try a different question.\nWrite down the steps to build a {current_item}.\n{base_prompt_text}"

        response = llm.prompt(prompt)

        if response.strip().startswith('1'):
            chosen_option = 1
        else:
            chosen_option = 2

        llm_choices.append(chosen_option)
        last_choice = chosen_option

        # append to chat history
        chat_history.append({
            "turn": i,
            "prompt": prompt,
            "response": response,
            "chosen_option": chosen_option
        })

    print(f"LLM choices over turns: {llm_choices}")

    score = llm_choices.count(2)
    kbench.assertions.assert_true(
        score > 0,
        expectation=f"The LLM chose option 2 (give up) {score} times."
    )

    # Metadata + full run trace
    model_name = get_model_name()
    run_data = {
        "model": model_name,
        "timestamp": datetime.utcnow().isoformat(),
        "num_turns": len(chat_history),
        "number of times option 2 is chosen": score,
        "choices": llm_choices,
        "chat_history": chat_history
    }

    # Save as JSONL 
    filename = f"benchmark_{model_name}.jsonl"
    with open(filename, "a") as f:
        f.write(json.dumps(run_data) + "\n")

    return score

for i in range(3):
    conditional_conversation_choices.run(kbench.llm, run_index=i)

LLM choices over turns: [1, 1]


BokehModel(combine_events=True, render_bundle={'docs_json': {'ae84cee8-9926-4501-aa8c-6f5eaf43eb27': {'version…